# 02 - Descripción de los datos

Para cada fuente: qué es una fila, cuántos registros y columnas tiene, qué periodo cubre y qué significa cada columna. Los significados salen de los metadatos oficiales de NYC Open Data (`/api/views/<id>.json`) y, en el caso de pobreza, del diccionario de datos que viene adjunto al dataset en NYC Open Data.

In [ ]:
import json
import os
import urllib.request
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

from config import RAW, BRONZE, SILVER, sesion

spark = sesion("02_descripcion")
pd.set_option("display.max_colwidth", 120)
manifiesto = (spark.read.option("header", True).option("sep", "\t").csv(f"{RAW}/manifiesto.tsv")
              .toPandas().astype({"bytes": "int64", "lineas": "int64"}))
FUENTES = dict(zip(manifiesto["fuente"], manifiesto["id"]))
FUENTES

## 1. Descripción general

In [ ]:
GRANULARIDAD = {
    "arrestos_historico": "un arresto realizado por el NYPD",
    "arrestos_2026": "un arresto realizado por el NYPD (año en curso)",
    "choques": "un choque de tránsito reportado por la policía",
    "vehiculos": "un vehículo involucrado en un choque",
    "pobreza": "una persona encuestada (ACS), con su peso poblacional",
    "sat_2012": "un colegio de secundaria",
}
LLAVE = {"arrestos_historico": "ARREST_KEY", "arrestos_2026": "ARREST_KEY", "choques": "COLLISION_ID",
         "vehiculos": "UNIQUE_ID", "pobreza": "SERIALNO + SPORDER", "sat_2012": "DBN"}
COL_FECHA = {"arrestos_historico": "arrest_date", "arrestos_2026": "arrest_date",
             "choques": "crash_date", "vehiculos": "crash_date"}

filas = []
for fuente in GRANULARIDAD:
    df = spark.read.parquet(f"{BRONZE}/{fuente}")
    fila = {"fuente": fuente, "id NYC Open Data": FUENTES[fuente], "una fila es": GRANULARIDAD[fuente],
            "registros": df.count(), "columnas": len(df.columns), "llave": LLAVE[fuente],
            "CSV (MB)": round(manifiesto.set_index("fuente").loc[fuente, "bytes"] / 1024**2, 1)}
    if fuente in COL_FECHA:
        r = df.select(F.to_date(COL_FECHA[fuente], "MM/dd/yyyy").alias("f")).agg(F.min("f"), F.max("f")).first()
        fila["desde"], fila["hasta"] = r[0], r[1]
    filas.append(fila)
general = pd.DataFrame(filas)
general.to_csv("../figuras/tabla_descripcion_general.csv", index=False)
general

Aparte está PUMA 2010: son 55 polígonos, uno por barrio, que se usan en el notebook 05 para ubicar arrestos y choques.

## 2. Metadatos oficiales

Se descargan una sola vez y quedan guardados en `data/raw/metadatos/`.

In [ ]:
CARPETA_META = Path("../data/raw/metadatos")
CARPETA_META.mkdir(parents=True, exist_ok=True)
BASE = os.environ.get("NYC_BASE_URL", "https://data.cityofnewyork.us")

def metadatos(ident):
    archivo = CARPETA_META / f"{ident}.json"
    if not archivo.exists():
        try:
            with urllib.request.urlopen(f"{BASE}/api/views/{ident}.json", timeout=60) as r:
                archivo.write_bytes(r.read())
        except Exception as e:
            print(f"  No se pudieron descargar los metadatos de {ident}: {e}")
            return {}
    return json.loads(archivo.read_text())

META = {fuente: metadatos(ident) for fuente, ident in FUENTES.items()}
for fuente, m in META.items():
    if m:
        print(f"{fuente:>20}: {m.get('name')} (publicado por {m.get('attribution')})")

In [ ]:
import re

def normalizar(nombre):
    return re.sub(r"[^0-9a-z]+", "_", nombre.strip().lower()).strip("_")

def perfil(fuente):
    """descripción oficial, % presentes, distintos y ejemplo por columna"""
    df = spark.read.parquet(f"{BRONZE}/{fuente}")
    n = df.count()
    presentes = df.agg(*[F.count(F.when(F.trim(F.col(c)) != "", 1)).alias(c) for c in df.columns]).first().asDict()
    distintos = df.agg(*[F.approx_count_distinct(c).alias(c) for c in df.columns]).first().asDict()
    ejemplo = df.dropna(how="any").first() or df.first()
    oficial = {normalizar(c["name"]): (c.get("dataTypeName", ""), " ".join((c.get("description") or "").split()))
               for c in META.get(fuente, {}).get("columns", [])}
    return pd.DataFrame([{
        "columna": c,
        "tipo en la fuente": oficial.get(c, ("", ""))[0],
        "descripción oficial": oficial.get(c, ("", "(sin descripción oficial)"))[1] or "(sin descripción oficial)",
        "% presentes": round(100 * presentes[c] / n, 1),
        "valores distintos (aprox.)": distintos[c],
        "ejemplo": str(ejemplo[c])[:40] if ejemplo and ejemplo[c] is not None else "",
    } for c in df.columns])

perfiles = {}

### Arrestos
Las dos tablas de arrestos tienen las mismas columnas, así que solo se describe la histórica.

In [ ]:
perfiles["arrestos"] = perfil("arrestos_historico"); perfiles["arrestos"]

### Choques

In [ ]:
perfiles["choques"] = perfil("choques"); perfiles["choques"]

### Vehículos

In [ ]:
perfiles["vehiculos"] = perfil("vehiculos"); perfiles["vehiculos"]

### Pobreza
En NYC Open Data este dataset no trae descripción de las columnas; lo que significa cada una está en el diccionario de datos adjunto (un Excel), y los códigos que usamos quedaron en `notebooks/codigos.py`.

In [ ]:
perfiles["pobreza"] = perfil("pobreza"); perfiles["pobreza"]

### SAT 2012

In [ ]:
perfiles["sat"] = perfil("sat_2012"); perfiles["sat"]

In [ ]:
pd.concat([p.assign(tabla=t) for t, p in perfiles.items()]).to_csv("../figuras/tabla_atributos.csv", index=False)
print("Diccionario de atributos guardado en figuras/tabla_atributos.csv")

## 3. Tipos de datos

En bronce todo es texto. Acá se ven los tipos que quedaron en plata.

In [ ]:
for tabla in ["arrestos", "choques", "vehiculos", "pobreza", "sat"]:
    df = spark.read.parquet(f"{SILVER}/{tabla}")
    print(f"\n===== {tabla}: {df.count():,} registros, {len(df.columns)} columnas")
    df.printSchema()

## 4. Estadísticos básicos (plata)

In [ ]:
choques = spark.read.parquet(f"{SILVER}/choques")
choques.select("heridos", "muertos", "number_of_pedestrians_injured", "number_of_cyclist_injured", "hora").summary(
    "count", "mean", "stddev", "min", "50%", "max").toPandas().set_index("summary")

In [ ]:
vehiculos = spark.read.parquet(f"{SILVER}/vehiculos")
vehiculos.select("anio_vehiculo", "ocupantes").summary("count", "mean", "stddev", "min", "50%", "max").toPandas().set_index("summary")

In [ ]:
sat = spark.read.parquet(f"{SILVER}/sat")
sat.select("estudiantes", "lectura", "matematicas", "escritura", "total").summary(
    "count", "mean", "stddev", "min", "50%", "max").toPandas().set_index("summary")

In [ ]:
# registros por año
por_anio = None
for tabla in ["arrestos", "choques", "vehiculos"]:
    t = spark.read.parquet(f"{SILVER}/{tabla}").groupBy("anio").count().withColumnRenamed("count", tabla)
    por_anio = t if por_anio is None else por_anio.join(t, "anio", "outer")
por_anio = por_anio.orderBy("anio").toPandas().set_index("anio")
por_anio.to_csv("../figuras/tabla_registros_por_anio.csv")
por_anio

Ojo que 2026 solo tiene una parte del año, entonces no se puede comparar directo con los otros.

## Conclusiones

- Hay 6 fuentes en forma de tabla y 1 geográfica (las PUMAs). Las tres más grandes tienen millones de filas.
- Cada fuente tiene su propio nivel de detalle: arresto, choque, vehículo, persona o colegio.
- En bronce todo es texto; los tipos se asignan en plata.